# RiskModels API — Quickstart (**Google Colab**)

**Local Jupyter?** Clone the repo and open [`riskmodels_quickstart.ipynb`](https://github.com/BlueWaterCorp/RiskModels_API/blob/main/notebooks/riskmodels_quickstart.ipynb) — from the repo root run `npm run ipynb`.

1. **Runtime → Run all** (the next cell installs `riskmodels-py` and `requests`).
2. **Secrets** (sidebar key icon): add **`RISKMODELS_API_KEY`**, or leave it blank and **paste when the setup cell prompts** (hidden where supported).

Get a key at [riskmodels.app/get-key](https://riskmodels.app/get-key).

In [ ]:
# Colab: RiskModels SDK + HTTP client (charts use matplotlib / numpy already on Colab)
%pip install -q "riskmodels-py>=0.3.2,<0.4" requests


In [ ]:
# ── Setup (Google Colab) ───────────────────────────────────────────────────────────────
from riskmodels import RiskModelsClient, rm, run
from riskmodels.aom import stock

import os
import requests
import pandas as pd
from IPython.display import display, Image

try:
    from google.colab import userdata
except ImportError:
    userdata = None


def _colab_api_key() -> str:
    for name in ("RISKMODELS_API_KEY", "RISKMODELS_QUICKSTART_API_KEY", "TEST_API_KEY"):
        v = (os.environ.get(name) or "").strip()
        if v and v != "PASTE_YOUR_KEY_HERE":
            print(f"Using {name} from the environment.")
            return v
    if userdata is not None:
        try:
            secret = userdata.get("RISKMODELS_API_KEY")
            if secret:
                s = str(secret).strip()
                os.environ["RISKMODELS_API_KEY"] = s
                print("Loaded RISKMODELS_API_KEY from Colab Secrets.")
                return s
        except Exception as e:
            print("Colab Secrets:", e)
    import getpass

    try:
        entered = getpass.getpass("RISKMODELS_API_KEY (hidden where supported): ").strip()
    except Exception:
        entered = input("RISKMODELS_API_KEY: ").strip()
    if not entered or entered == "PASTE_YOUR_KEY_HERE":
        raise ValueError("RISKMODELS_API_KEY required. https://riskmodels.app/get-key")
    os.environ.setdefault("RISKMODELS_API_KEY", entered)
    return entered


API_KEY = _colab_api_key()

_orig = (os.environ.get("RISKMODELS_QUICKSTART_BASE_URL") or os.environ.get("RISKMODELS_API_BASE_URL") or "https://riskmodels.app").strip()
_orig = _orig.removesuffix("/api").rstrip("/")
BASE_URL = _orig + "/api"

session = requests.Session()
session.headers["Authorization"] = f"Bearer {API_KEY}"

r = session.get(f"{BASE_URL}/balance", params={"include_usage": "false"})
r.raise_for_status()
_prefix = (API_KEY or "")[:16]
print(f"Connected. Key: {_prefix}...")

os.environ.setdefault("RISKMODELS_BASE_URL", BASE_URL)

client = RiskModelsClient.from_env()
print("SDK ready: client, rm, run, stock.")

---
## Precision Hedge Chart

See how a stock's return decomposes into market, sector, and subsector factor legs over time.

In [ ]:
# ── Precision Hedge Chart ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

TICKER = "NVDA"   # Try it: change to "AAPL", "TSLA", or "JPM"
YEARS  = 3        # Try it: set to 1 or 5 to zoom in/out

resp = session.get(
    f"{BASE_URL}/ticker-returns",
    params={"ticker": TICKER, "years": YEARS},
)
resp.raise_for_status()
data = resp.json().get("data", [])
df = pd.DataFrame(data)

if df.empty:
    print(f"No data for {TICKER}.")
else:
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    def cumulative_decimal(r: pd.Series) -> np.ndarray:
        x = np.asarray(r.fillna(0.0), dtype=float)
        return np.cumprod(1.0 + x) - 1.0

    df["cum_stock"]     = cumulative_decimal(df["returns_gross"])
    df["cum_market"]    = cumulative_decimal(df["l1_cfr"])
    df["cum_sector"]    = cumulative_decimal(df["l2_cfr"])
    df["cum_subsector"] = cumulative_decimal(df["l3_cfr"])

    fig, ax = plt.subplots(figsize=(12, 5))
    fig.patch.set_facecolor("#0d0d0d")
    ax.set_facecolor("#0d0d0d")

    colors = {
        "cum_stock":     "#60a5fa",
        "cum_market":    "#6366f1",
        "cum_sector":    "#34d399",
        "cum_subsector": "#9ca3af",
    }
    labels = {
        "cum_stock":     TICKER,
        "cum_market":    "Market (L1)",
        "cum_sector":    "Sector (L2)",
        "cum_subsector": "Subsector (L3)",
    }

    for col, color in colors.items():
        ax.plot(df["date"], df[col], color=color, linewidth=1.4, label=labels[col])

    ax.axhline(0, color="#444", linewidth=0.8, linestyle="--")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0, decimals=0))
    ax.tick_params(colors="#aaa", labelsize=9)
    for spine in ax.spines.values():
        spine.set_edgecolor("#333")
    ax.set_xlabel("Date", color="#aaa", fontsize=9)
    ax.set_ylabel("Cumulative Return", color="#aaa", fontsize=9)
    ax.set_title(
        f"Your Precision Hedge Recipe — {TICKER}  ({YEARS}y)",
        color="white", fontsize=12, pad=10
    )
    ax.legend(
        frameon=False, labelcolor="#ccc", fontsize=9,
        loc="upper left", title="Series", title_fontsize=8,
    )
    ax.grid(axis="y", color="#222", linewidth=0.6)
    plt.tight_layout()
    plt.show()

    latest = df.iloc[-1]
    summary = pd.DataFrame({
        "Value": {
            f"{TICKER} total return":    f"{latest.cum_stock * 100:.1f}%",
            "Market (L1 CFR)":           f"{latest.cum_market * 100:.1f}%",
            "Sector (L2 CFR)":           f"{latest.cum_sector * 100:.1f}%",
            "Subsector (L3 CFR)":        f"{latest.cum_subsector * 100:.1f}%",
            "Residual (approx.)": f"{(latest.cum_stock - latest.cum_subsector) * 100:.1f}%",
        }
    })
    print(f"Cumulative returns over {YEARS}y — as of {latest.date.date()}")
    display(summary)

---
## Risk Snapshot — Hedge Ratios & Volatility

One API call gives you hedge ratios at three precision levels, plus volatility and explained risk.

In [ ]:
# ── Risk Snapshot ────────────────────────────────────────────────────────────────────
ticker = "NVDA"  # Try it: change to "AAPL", "TSLA", "JPM"

resp = session.get(f"{BASE_URL}/metrics/{ticker}")
resp.raise_for_status()
m = resp.json()
metrics = m.get("metrics", m)
meta = m.get("meta", {})

snapshot = pd.DataFrame({"Value": {
    "Date":             m.get("teo"),
    "Price":            metrics.get("price_close"),
    "Vol (23d)":        round(metrics["vol_23d"], 4) if metrics.get("vol_23d") else None,
    "Sector ETF":       meta.get("sector_etf"),
    "Subsector ETF":    meta.get("subsector_etf"),
    "L1 Market HR":     round(metrics["l1_mkt_hr"], 4) if metrics.get("l1_mkt_hr") else None,
    "L1 Market ER":     round(metrics["l1_mkt_er"], 4) if metrics.get("l1_mkt_er") else None,
    "L2 Market HR":     round(metrics["l2_mkt_hr"], 4) if metrics.get("l2_mkt_hr") else None,
    "L2 Sector HR":     round(metrics["l2_sec_hr"], 4) if metrics.get("l2_sec_hr") else None,
    "L3 Market HR":     round(metrics["l3_mkt_hr"], 4) if metrics.get("l3_mkt_hr") else None,
    "L3 Sector HR":     round(metrics["l3_sec_hr"], 4) if metrics.get("l3_sec_hr") else None,
    "L3 Subsector HR":  round(metrics["l3_sub_hr"], 4) if metrics.get("l3_sub_hr") else None,
    "L3 Residual ER":   round(metrics["l3_res_er"], 4) if metrics.get("l3_res_er") else None,
}})
print(f"{ticker} — Risk Snapshot")
display(snapshot)

---
## Deep Dive Snapshot

A precomputed risk report as a PNG image — one API call, no rendering code.

In [ ]:
# ── Deep Dive Snapshot ──────────────────────────────────────────────────────────────
ticker = "NVDA"  # Try it: change to "AAPL", "MSFT", "GOOGL"

resp = session.get(f"{BASE_URL}/snapshot/{ticker}", params={"format": "png"})
if resp.status_code == 404:
    print(f"No snapshot for {ticker} (not yet in coverage).")
else:
    resp.raise_for_status()
    display(Image(data=resp.content))

---
## Portfolio Hedge

Submit weighted positions and get back hedge ratios at every level in one batch call.

In [ ]:
# ── Portfolio Hedge ──────────────────────────────────────────────────────────────────
# Try it: change tickers and weights to your own portfolio (weights should sum to 1.0)
portfolio = {
    "AAPL":  0.25,
    "MSFT":  0.20,
    "NVDA":  0.20,
    "GOOGL": 0.15,
    "AMZN":  0.10,
    "JPM":   0.10,
}

resp = session.post(
    f"{BASE_URL}/batch/analyze",
    json={"tickers": list(portfolio.keys()), "metrics": ["hedge_ratios"], "years": 1},
)
resp.raise_for_status()
results = resp.json()["results"]

rows = []
for ticker, weight in portfolio.items():
    r  = results.get(ticker, {})
    hr = r.get("hedge_ratios") or {}
    rows.append({
        "ticker":       ticker,
        "weight":       weight,
        "l1_market_hr": hr.get("l1_market"),
        "l2_market_hr": hr.get("l2_market"),
        "l2_sector_hr": hr.get("l2_sector"),
        "l3_market_hr": hr.get("l3_market"),
        "l3_sector_hr": hr.get("l3_sector"),
        "l3_sub_hr":    hr.get("l3_subsector"),
    })

df_port = pd.DataFrame(rows).set_index("ticker")

def _wtd(col):
    return round((df_port["weight"] * df_port[col].fillna(0)).sum(), 4)

port_summary = pd.DataFrame({
    "Market HR": [_wtd("l1_market_hr"), _wtd("l2_market_hr"), _wtd("l3_market_hr")],
    "Sector HR": [float("nan"), _wtd("l2_sector_hr"), _wtd("l3_sector_hr")],
    "Subsector HR": [float("nan"), float("nan"), _wtd("l3_sub_hr")],
}, index=["L1", "L2", "L3"]).rename_axis("Level")

print("Portfolio-level hedge ratios (weighted):")
display(port_summary)

print("\nPer-position breakdown:")
display(df_port[["weight", "l1_market_hr", "l2_market_hr",
                  "l2_sector_hr", "l3_market_hr", "l3_sector_hr", "l3_sub_hr"]])

---
## Rankings — Where Does Your Stock Rank?

Cross-sectional percentile rankings across the full universe, sector, or subsector.

In [ ]:
# ── Rankings ─────────────────────────────────────────────────────────────────────────
ticker = "NVDA"  # Try it: change to any ticker

resp = session.get(f"{BASE_URL}/rankings/{ticker}")
resp.raise_for_status()
data = resp.json()

if data.get("rankings"):
    df_rank = pd.DataFrame(data["rankings"])
    print(f"{data['ticker']} — Rankings (as of {data.get('date', 'latest')})\n")
    display(df_rank[["display_label", "rank_percentile", "rank_ordinal", "cohort_size"]].head(15))
else:
    print(f"No rankings for {ticker}")

---
## Factor Risk Decomposition

Break down a stock's risk into market, sector, subsector, and residual components month by month.

In [ ]:
# ── Factor Risk Table ────────────────────────────────────────────────────────────────
ticker = "NVDA"  # Try it: change to any ticker

resp = session.get(
    f"{BASE_URL}/l3-decomposition",
    params={"ticker": ticker, "market_factor_etf": "SPY"},
)
resp.raise_for_status()
body = resp.json()

df_risk = pd.DataFrame({
    "date":         pd.to_datetime(body.get("dates", [])),
    "market_er":    body.get("l3_market_er", []),
    "sector_er":    body.get("l3_sector_er", []),
    "subsector_er": body.get("l3_subsector_er", []),
    "residual_er":  body.get("l3_residual_er", []),
})
df_risk[["market_er", "sector_er", "subsector_er", "residual_er"]] = (
    df_risk[["market_er", "sector_er", "subsector_er", "residual_er"]].fillna(0)
)
df_risk = df_risk.sort_values("date").reset_index(drop=True)

if df_risk.empty:
    print(f"No factor data for {ticker}.")
else:
    df_risk["total_er"] = df_risk[["market_er", "sector_er",
                                    "subsector_er", "residual_er"]].sum(axis=1)

    df_risk["month"] = df_risk["date"].dt.to_period("M")
    df_month = df_risk.groupby("month", as_index=False).last().drop(columns=["month"])

    pct_cols = ["market_er", "sector_er", "subsector_er", "residual_er", "total_er"]
    df_month[pct_cols] = (df_month[pct_cols] * 100).round(2)
    df_month.rename(columns={
        "market_er": "market_%",
        "sector_er": "sector_%",
        "subsector_er": "subsector_%",
        "residual_er": "residual_%",
        "total_er": "total_%",
    }, inplace=True)

    print(f"Factor risk attribution for {ticker} — most recent 12 months (month-end, L3 ER → %)")
    print(f"Market ETF: {body.get('market_factor_etf', '—')}  |  Universe: {body.get('universe', '—')}")
    print()
    display(df_month.tail(12))

---
## Next Steps

- **API docs:** [riskmodels.app/docs/api](https://riskmodels.app/docs/api)
- **Python SDK:** `pip install riskmodels-py` — wraps every endpoint above into one-liners
- **Questions?** [service@riskmodels.app](mailto:service@riskmodels.app)

---
## SDK L3 Decomposition Visual

Pass the raw `/l3-decomposition` API JSON directly into the SDK visual. The chart uses exact API fields such as `l3_market_er`, `l3_sector_er`, `l3_subsector_er`, and `l3_residual_er`; the only abstraction is color.

In [ ]:
# ── SDK L3 Decomposition Visual ─────────────────────────────────────────────────────
# If needed in a fresh notebook runtime, uncomment the next line:
# %pip install -q "riskmodels-py[viz]"

from riskmodels import plot_l3_decomposition

sdk_ticker = "NVDA"  # Try it: change to any ticker

resp = session.get(
    f"{BASE_URL}/l3-decomposition",
    params={"ticker": sdk_ticker, "market_factor_etf": "SPY", "years": 1},
)
resp.raise_for_status()
raw_l3 = resp.json()

fig = plot_l3_decomposition(
    raw_l3,
    metric="variance",
    mode="timeseries",
    title=f"{sdk_ticker} L3 variance decomposition — exact API fields",
)
fig.show()

print("Exact L3 mapping used by the SDK visual:")
display(pd.DataFrame(
    fig.layout.meta["l3_mapping"].items(),
    columns=["L3 layer", "API field"],
))